# Phase 3 — Logistic Regression Baseline

This notebook builds the first FraudShield AI classifier and measures performance on a later time period. It is intentionally a **baseline**, not a tuned final model.

## Evaluation design

- Sort chronologically by `Time`; earlier 60% for training, the next 20% for validation, and the final 20% for test (adjusting boundaries so equal timestamps are not split).
- Fit the median imputer and `StandardScaler` using training data only, inside an sklearn `Pipeline`.
- Use the original class distribution in validation and test. Do not apply SMOTE or undersampling to those sets.
- Use threshold `0.5` as a reference only. We will optimize decisions later using validation data; do not tune against the test set.

The source data is heavily imbalanced, and `V1`–`V28` are anonymized PCA components.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

# This notebook expects to be started from the repository root.
ROOT = Path.cwd()
if not (ROOT / "configs" / "config.yaml").exists():
    raise FileNotFoundError("Start Jupyter from the fraudshield-ai project root.")
DATA_PATH = ROOT / "data" / "raw" / "creditcard.csv"
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"


## 1. Inspect the dataset and target distribution

First verify that the actual file still has the expected schema. Do not assume that downloaded data is identical to the previously analyzed file.

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())
display(df.dtypes.value_counts().rename_axis("dtype").to_frame("column_count"))

class_counts = df["Class"].value_counts().sort_index()
class_summary = pd.DataFrame({"count": class_counts, "percentage": class_counts / len(df) * 100})
display(class_summary)
print("Exact duplicate extra copies:", int(df.duplicated().sum()))
print("Missing cells:", int(df.isna().sum().sum()))


## 2. Create chronological partitions

This split is designed to test later transactions rather than randomly mixing earlier and later records. Since fraud prevalence changes over time, inspect the positive count in every partition before interpreting metrics.

In [ ]:
from src.data.splitting import chronological_train_validation_test_split

splits = chronological_train_validation_test_split(df, time_column="Time", train_fraction=0.60, validation_fraction=0.20)

def summarize_partition(name, part):
    return {
        "partition": name,
        "rows": len(part),
        "fraud_count": int(part["Class"].sum()),
        "legitimate_count": int((part["Class"] == 0).sum()),
        "fraud_rate_percent": part["Class"].mean() * 100,
        "time_min": part["Time"].min(),
        "time_max": part["Time"].max(),
    }

split_summary = pd.DataFrame([
    summarize_partition("train", splits.train),
    summarize_partition("validation", splits.validation),
    summarize_partition("test", splits.test),
]).set_index("partition")
display(split_summary)
assert splits.train["Time"].max() < splits.validation["Time"].min()
assert splits.validation["Time"].max() < splits.test["Time"].min()


## 3. Train the baseline

The function below creates a pipeline containing median imputation, standardization, and Logistic Regression. Because all learned preprocessing is inside the pipeline, `fit` learns preprocessing parameters from the training partition only. The model does not use `Class` as an input feature.

In [ ]:
from src.models.baseline import run_baseline

report = run_baseline(data_path=DATA_PATH, output_dir=ROOT / "reports" / "baseline", model_path=ROOT / "models" / "logistic_regression_baseline.joblib", threshold=0.5)
print("Baseline trained and saved.")


## 4. Compare metrics

- **Precision:** among flagged transactions, the fraction that are fraud.
- **Recall:** among all fraud transactions, the fraction detected.
- **F1:** harmonic mean of precision and recall.
- **ROC-AUC:** ranking performance across classification thresholds.
- **Average Precision (AP):** primary precision-recall ranking summary; this is often called PR-AUC, but it differs slightly from trapezoidal PR-AUC, so both are reported.
- **Confusion matrix:** true negatives, false positives, false negatives, and true positives.

Accuracy can be very high even when a model misses many fraud cases, so do not use it alone.

In [ ]:
metrics_path = ROOT / "reports" / "baseline" / "baseline_metrics.json"
with metrics_path.open(encoding="utf-8") as f:
    report = json.load(f)

metrics_table = pd.DataFrame([
    {"partition": name, **values}
    for name, values in report["metrics"].items()
]).set_index("partition")
columns = ["n_samples", "n_positive_actual", "accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc_average_precision", "pr_auc_trapezoidal", "true_positive", "false_positive", "false_negative", "true_negative"]
display(metrics_table[columns].style.format({
    "accuracy": "{:.4%}", "precision": "{:.4f}", "recall": "{:.4f}", "f1": "{:.4f}",
    "roc_auc": "{:.4f}", "pr_auc_average_precision": "{:.4f}", "pr_auc_trapezoidal": "{:.4f}"
}))


## 5. Inspect the validation and test charts

Keep validation and test separate: validation is available for decisions in later phases; the test set should remain an unbiased final check of the chosen approach.

In [ ]:
for filename in [
    "confusion_matrix_validation.png",
    "roc_curve_validation.png",
    "precision_recall_curve_validation.png",
    "confusion_matrix_test.png",
    "roc_curve_test.png",
    "precision_recall_curve_test.png",
]:
    path = ROOT / "reports" / "baseline" / filename
    print(filename)
    display(Image(filename=str(path)))


## 6. Interpretation and next step

The `0.5` threshold is a reference, not a business-optimal choice. With the original data imbalance, recall may be insufficient even when ROC-AUC is high. In Phase 4 we will examine imbalance-handling methods and later choose a threshold from validation results. Do not tune or select a method based on repeated test-set inspection.